# Module 9 — Audit metrics, code, and claims

For each example:
1. Predict the result from the business requirement.
2. Run the original code and identify the problematic line or calculation.
3. Explain why it conflicts with the requirement.
4. Write and run corrected code in the cell provided below the original.
5. Compare with your prediction and test a boundary case or a change that should not affect the result.

Keep the original code unchanged so you can compare. Record your reasoning before opening the worked notebook. This class notebook is not an additional homework submission.

Run `uv sync --frozen` and select this repository's `.venv` kernel.


In [ ]:
import pandas as pd
import random
from itertools import permutations

print("Ready for the four examples.")

## 1 · Customer loyalty
**Request:** “Show which store has the most loyal customers.”

Before coding, ask: Who uses the result? What decision follows? Does loyalty mean frequent orders, repeat visits, spending, or something else?

For this example, all rows are distinct completed orders in January 2026. Customer IDs are reliable; every order belongs to one store. We compare repeat purchasing during this month, not long-run retention or a causal effect of the store.

In [ ]:
orders = pd.DataFrame({
    "order": ["o1", "o2", "o3", "o4", "o5", "o6"],
    "store": ["A", "A", "A", "A", "B", "B"],
    "customer": ["Ana", "Ana", "Ana", "Ben", "Cai", "Cai"],
    "date": ["2026-01-02", "2026-01-03", "2026-01-04",
             "2026-01-02", "2026-01-02", "2026-01-05"]
})
print(orders.to_string(index=False))

### Pause and predict
1. Calculate each store's average orders per customer.
2. Calculate the share of its customers with at least two orders.
3. Do these measures rank the stores in the same way?
4. Could a newly acquired customer be less likely to qualify simply because they joined late?

Use `talk-it-through`, if available, to discuss the choices one question at a time. Without the skill, ask an agent: “Help me define this metric before implementing it. Ask one question at a time about the decision, population, denominator, time window, and assumptions. Ask me to predict a tiny example before recommending a definition. Summarize our agreement before coding.”

**Your prediction and reasoning:**



In [ ]:
per_customer = (orders.groupby(["store", "customer"])
                .size().reset_index(name="n_orders"))
print(per_customer.to_string(index=False))

### Run and inspect
Predict the two store outputs, then run the code below.

**Your prediction and reasoning:**



In [ ]:
repeat = per_customer[per_customer["n_orders"] >= 2]
wrong_share = repeat.groupby("store").size() / orders.groupby("store").size()
print(wrong_share)

### Identify and explain
- Problematic line or calculation:
- Why it conflicts with repeat-customer share:

### Correct
Complete the function below. Return one share per store with at least one order, including stores with no repeat customers. Use the supplied completed orders; do not hard-code store names or answers.


In [ ]:
def repeat_customer_share(order_rows):
    # Write your corrected calculation.
    pass

# Call your function on orders and display its result.


### Verify and test a boundary case
Compare your result with the original prediction. Add store C with two customers who each placed one order. Predict its share, then run your function on the expanded data. Explain why a store with no orders is different.

**Expected versus actual results and explanation:**


In [ ]:
# Create the expanded orders table without changing orders.
# Run your function and compare with your prediction.


## 2 · Average tip per visit
**Requirement:** report average tip per visit. A blank tip explicitly means the customer left no tip. Each visit has at least one service.

**Pause and predict:** Visit A has a \$10 tip and two services; visit B has no tip and one service. What is the mean tip per visit? What happens if we calculate the mean after the merge?

**Your prediction and reasoning:**



In [ ]:
visits = pd.DataFrame({"visit": ["A", "B"], "tip": [10.0, None]})
services = pd.DataFrame({"visit": ["A", "A", "B"],
                         "service": ["haircut", "coloring", "haircut"]})
print("Visits:\n", visits.to_string(index=False))
print("\nServices:\n", services.to_string(index=False))

In [ ]:
clean_visits = visits.copy()
clean_visits["tip"] = clean_visits["tip"].fillna(0)
merged = services.merge(clean_visits, on="visit", validate="many_to_one")
print(merged.to_string(index=False))
wrong_mean_tip = merged["tip"].mean()
print("Mean after merge:", round(wrong_mean_tip, 2))

### Identify and explain
- Problematic line or calculation:
- What one merged row represents, and why that matters:

### Correct
Write a function that returns average tip per visit. Under the stated rule, missing tips count as zero. Preserve the original code and tables.


In [ ]:
def average_tip_per_visit(visit_rows):
    # Write your corrected calculation.
    pass

# Call your function on visits and display its result.


### Verify and test a change that should not affect the result
Compare your corrected mean with your prediction. Add a third service to visit A without changing the visits. Recompute the original merged-row mean and your corrected visit mean. Which result should stay unchanged, and why?

**Expected versus actual results and explanation:**


In [ ]:
# Add a service in a new services table.
# Recompute the original and corrected means and compare.


### Change the assumption
What if a blank tip means not recorded rather than no tip? State what you can calculate and what remains unknown.

**Your reasoning:**


## 3 · Delivery performance
**Request:** “Show our overall on-time delivery rate.”

For the numeric example, each total counts completed deliveries with known outcomes, and each delivery belongs to exactly one store. The on-time rule is already fixed. Later, we will question those assumptions.

In [ ]:
delivery = pd.DataFrame({"store": ["A", "B"],
                         "on_time": [9, 1], "total": [10, 2]})
print(delivery.to_string(index=False))

### Pause and predict
1. What is each store's on-time rate?
2. What is the average of those two rates?
3. What share of all deliveries were on time?
4. Which result gives each store equal weight? Which gives each delivery equal weight?

**Your prediction and reasoning:**



In [ ]:
delivery["rate"] = delivery["on_time"] / delivery["total"]
overall_delivery_rate = delivery["rate"].mean()
print(delivery.to_string(index=False))
print(f"Overall delivery rate: {overall_delivery_rate:.2%}")


### Identify and explain
- Problematic line or calculation:
- What gets equal weight in this calculation, and why that conflicts with the request:

### Correct
Write a function that returns the share of all deliveries that were on time. Use the counts provided; do not hard-code the result.


In [ ]:
def overall_on_time(delivery_rows):
    # Write your corrected calculation.
    pass

# Call your function on delivery and display its result.


### Verify independently
Represent the twelve deliveries as individual on-time or late outcomes. Compute their on-time share and compare it with your corrected function.

**Expected versus actual results and explanation:**


In [ ]:
# Write an independent check using individual delivery outcomes.


### New check before running
Suppose A's records are split into two batches, each with 5 deliveries: one with 5 on-time deliveries and one with 4. B is unchanged. Does the overall delivery rate change? Should it?

### Test a change that should not affect the result
Predict the result after splitting store A. Run your corrected function on the batches below. Compare it with the unweighted average of batch rates and explain any difference.

**Prediction, actual result, and explanation:**


In [ ]:
batches = pd.DataFrame({"batch": ["A1", "A2", "B"],
                        "on_time": [5, 4, 1], "total": [5, 5, 2]})
# Your comparison

## 4 · A simplified weighted lottery
This is a hypothetical lottery, not a model of current immigration policy.

Person A holds 1 entry and person B holds 4 entries. Shuffle all entries uniformly. Scan the line, keeping only each person's first entry, until the available slots are filled. A person can win at most once. Slots never exceed the number of people.

An app uses `slots × person's entries / total entries` as the person's probability of selection.

### Pause and predict
With one slot, what are the two probabilities? With two slots? Can a probability exceed 1?

**Your prediction and reasoning:**



In [ ]:
def linear_odds(weights, slots):
    return [slots * weight / sum(weights) for weight in weights]

print("One slot:", linear_odds([1, 4], 1))
print("Two slots:", linear_odds([1, 4], 2))

### Identify and explain
- Problematic line or calculation:
- Why multiplying the one-slot probability by the number of slots fails:
- Would clipping the result to 1 fix the selection rule? Explain.

### Correct
Replace the formula with a simulation of the stated mechanism. Outline your algorithm before coding. Return estimated selection probabilities in the same order as `weights`.

Assume positive integer weights, an integer number of slots from zero through the number of people, and a positive number of runs. Use a fixed seed. Do not hard-code the two-person answer. You may ask for a hint after outlining your approach.

**My algorithm:**


In [ ]:
def selection_chances(weights, slots, runs=5000, seed=576):
    # Simulate the stated ticket mechanism.
    # Each person can be selected at most once per run.
    # Return a list of estimated probabilities.
    pass

# Call your function for weights [1, 4] with one slot and two slots.


### Verify and test boundaries
Compare your estimates with the one-slot and two-slot probabilities you predicted. Then test zero slots, enough slots for everyone, and equal weights with one slot. Predict each result before running it.

Check that every probability lies between 0 and 1 and that their sum equals the number of slots (allowing floating-point tolerance). Simulation estimates need not match theoretical probabilities exactly. Use exact comparisons only for outcomes that are guaranteed by the mechanism.

**Expected versus actual results:**

**What these checks establish, and what remains untested:**


In [ ]:
# Run your boundary checks here and display the results.


### Optional extension
The worked notebook enumerates all permutations of five labeled tickets to check the simulation exactly. Read this only after predicting the tiny case; enumeration is practical here because the example is small.


## Apply the audit
Choose one number in your team's app:
1. What exactly should it measure?
2. What tiny case would help you check it?
3. Which assumption or line deserves inspection?
4. What evidence would justify a change?
